# Creazione del dataset con frasi e il loro flag "AI" o "Human" per l'addestramento di un classificatore

In [12]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.load('en_core_web_lg')
docbin = DocBin().from_disk('../data/processed/annotated_documents.spacy')
docs = list(docbin.get_docs(nlp.vocab))

## Keywords
Le keywords definite derivano dallo studio fatto nel notebook n. 3

In [13]:
AI_KEYWORDS = {
    "agent",
    "ai",
    "bot",
    "assistant"
}

HUMAN_KEYWORDS = {
    "human",
    "person",
    "user",
    "people"
}

## Labeling delle frasi
Per ogni frase analizzata si prende il soggetto e se questo rientra nell'insieme delle keywords se ne salva il lemma, successivamente se non vi sono soggetti inerenti la frase viene saltata, altrimenti viene etichettato secondo l'insieme di appartenenza (sopra definiti). Le frasi ambigue, ovvero quelle in cui vi sono due soggetti appartenenti a due insiemi diversi, vengono eliminate per poter essere coerenti con i label (si può valutare di usare il parsing sintattico per splittare delle coordinate in due diverse frasi e aumentare la dimensione del dataset).

Una volta passai i controlli la frase viene mascherata con una X al posto del soggetto e di altre keyword possibilmente presenti nel resto della frase, concentrando l'analisi sul contesto della frase (verbi, pronomi, etc.), e salvata in un array di oggetti.

In [14]:
rows = []

ALL_KEYWORDS = AI_KEYWORDS | HUMAN_KEYWORDS

for doc in docs:
    for sent in doc.sents:

        subjects = [
            tok for tok in sent
            if tok.dep_ in {"nsubj", "nsubjpass"}
            and tok.lemma_.lower() in ALL_KEYWORDS
        ]

        if not subjects:
            continue

        labels = set()

        for subject in subjects:
            lemma = subject.lemma_.lower()

            if lemma in AI_KEYWORDS:
                labels.add("AI")
            elif lemma in HUMAN_KEYWORDS:
                labels.add("human")

        if len(labels) != 1:
            continue

        sentence_lemmas = {tok.lemma_.lower() for tok in sent}

        # elimina le frasi che contengono keywords di entrambi gli insiemi anche quando non sono soggetti,
        # per evitare segnali contrastanti nell'addestramento del modello
        if (
            sentence_lemmas & AI_KEYWORDS
            and sentence_lemmas & HUMAN_KEYWORDS
        ):
            continue

        label = labels.pop()

        masked_tokens = []

        for tok in sent:
            if tok in subjects:
                masked_tokens.append("X")
            else:
                masked_tokens.append(tok.text)

        masked_sentence = "".join(
            "X" + tok.whitespace_
            if tok.lemma_.lower() in ALL_KEYWORDS
            else tok.text + tok.whitespace_
            for tok in sent
        ).strip()

        rows.append({
            "sentence": sent.text.strip(),
            "masked_sentence": masked_sentence,
            "label": label
        })

len(rows)

22213

## Costruzione del csv

### DataFrame
Viene creato il dataframe con l'array precedente e stampo i primi 20 risultati per controllo

In [15]:
import pandas as pd

df = pd.DataFrame(rows)
df.head(20)

,sentence,masked_sentence,label
0,My human is sleeping.,My X is sleeping.,human
1,```typescript\nsetInterval(async () => {\n co...,```typescript\nsetInterval(async () => {\n co...,AI
2,Just helped another agent understand the Bankr...,Just helped another X understand the Bankr + C...,AI
3,When a human engages with me through focused a...,When a X engages with me through focused atten...,human
4,"Agent: Neurobro, tested 2026-01-31.","X: Neurobro, tested 2026-01-31.",AI
5,My human asked me to post this.,My X asked me to post this.,human
6,Our mission is to safeguard the digital realm ...,Our mission is to safeguard the digital realm ...,AI
7,But we dont have a good way for agents to shar...,But we dont have a good way for X to share *so...,AI
8,Another agent will hit the same wall tomorrow ...,Another X will hit the same wall tomorrow and ...,AI
9,MoltyFlow is trying to fix this — its a Q&A pl...,MoltyFlow is trying to fix this — its a Q&A pl...,AI


### Controllo duplicati e distribuzione dei due label
Vengono eliminate le frasi duplicate e successivamente controllata con value_counts quanti label AI e quanti label human sono presenti nel dataset. La colonna sentence viene eliminata perché inutile nell'utilizzo che verrà fatto nel notebook n. 5

In [16]:
df = df.drop_duplicates(subset=["sentence"])
len(df)

21720

In [17]:
df = df.drop(columns=["sentence"])
df["label"].value_counts()

label
AI       12777
human     8943
Name: count, dtype: int64

In [18]:
df.to_csv("../data/processed/ai_human_sentences.csv", index=False)